# Tanseek — Connected data & conflict-engine starter

This notebook uses the **existing synthetic CSVs** that follow the PostgreSQL schema. It builds relational views and a deterministic constraint checker as a starting point for Menna's scheduling model. Run top to bottom. Keep the `Tanseek_CSV_Data` folder and `Tanseek_constraint_cases.csv` beside this notebook. No database or internet is required.

Policy: Saturday–Wednesday, 09:00–17:00, four contiguous 120-minute slots. Availability is due August 31 each year. Data is synthetic; dates in this fixture are for the 2027 term.

In [ ]:
from pathlib import Path
from collections import defaultdict
import pandas as pd

ROOT = Path.cwd()
DATA = ROOT / "Tanseek_CSV_Data"
CASES = ROOT / "Tanseek_constraint_cases.csv"
assert DATA.is_dir() and CASES.is_file(), "Extract the ZIP beside this notebook first."
tables = {p.stem: pd.read_csv(p, encoding="utf-8-sig") for p in DATA.glob("*.csv")}
cases = pd.read_csv(CASES, encoding="utf-8-sig").fillna("")
pd.DataFrame([(name, len(frame), ", ".join(frame.columns)) for name, frame in sorted(tables.items())], columns=["table", "rows", "columns"])


## Relationship checks

These verify the foreign keys needed by the model. A missing reference raises an error before a schedule is evaluated.

In [ ]:
relations = [
    ("courses", "department_id", "departments", "id"),
    ("sections", "course_id", "courses", "id"),
    ("sections", "term_id", "academic_terms", "id"),
    ("section_groups", "section_id", "sections", "id"),
    ("section_groups", "group_id", "student_groups", "id"),
    ("section_instructors", "section_id", "sections", "id"),
    ("section_instructors", "instructor_id", "accounts", "id"),
    ("section_instructors", "requirement_id", "session_requirements", "id"),
    ("required_equipment", "requirement_id", "session_requirements", "id"),
    ("room_equipment", "room_id", "rooms", "id"),
    ("availability_slots", "submission_id", "availability_submissions", "id"),
    ("availability_slots", "slot_id", "time_slots", "id"),
    ("allocations", "section_id", "sections", "id"),
    ("allocations", "start_slot_id", "time_slots", "id"),
]
integrity = []
for child, column, parent, key in relations:
    missing = set(tables[child][column].dropna()) - set(tables[parent][key])
    integrity.append((child + "." + column, parent + "." + key, len(missing)))
integrity = pd.DataFrame(integrity, columns=["foreign_key", "references", "missing_ids"])
assert (integrity.missing_ids == 0).all(), integrity[integrity.missing_ids > 0]
integrity


## Connected views

Aggregate many-to-many links *before* joining them to sections, so group counts and equipment quantities do not get multiplied. `section_view` gives one row per section. `candidate_view` gives one row per eligible section/instructor/requirement/room/slot combination after the hard checks below.

In [ ]:
group_links = tables["section_groups"].merge(
    tables["student_groups"][["id", "student_count", "name"]], left_on="group_id", right_on="id", validate="many_to_one"
)
group_summary = group_links.groupby("section_id", as_index=False).agg(
    group_ids=("group_id", list), student_count=("student_count", "sum"), group_names=("name", list)
)
section_view = (tables["sections"].merge(tables["courses"][["id", "code", "title", "department_id"]],
             left_on="course_id", right_on="id", validate="many_to_one", suffixes=("", "_course"))
             .merge(group_summary, left_on="id", right_on="section_id", validate="one_to_one", suffixes=("", "_groups")))
assert len(section_view) == len(tables["sections"])
section_view[["id", "code", "group_names", "student_count", "department_id"]].head()


In [ ]:
# Compact maps keep the checker readable, while the DataFrames above remain available for analysis.
def by_id(name): return tables[name].set_index("id").to_dict("index")
sections, courses, requirements = [by_id(n) for n in ("sections", "courses", "session_requirements")]
rooms, slots, accounts = [by_id(n) for n in ("rooms", "time_slots", "accounts")]
groups_by_section = {int(r.section_id): set(map(int, r.group_ids)) for r in group_summary.itertuples()}
students_by_section = {int(r.section_id): int(r.student_count) for r in group_summary.itertuples()}
required = {(int(r.requirement_id), int(r.equipment_id)): int(r.quantity) for r in tables["required_equipment"].itertuples()}
available = {(int(r.room_id), int(r.equipment_id)): int(r.quantity) for r in tables["room_equipment"].itertuples()}
eligible = {(int(r.section_id), int(r.requirement_id), int(r.instructor_id)) for r in tables["section_instructors"].itertuples()}
submissions = tables["availability_submissions"].sort_values(["revision", "id"]).drop_duplicates(["term_id", "instructor_id"], keep="last")
submission_by_staff = {(int(r.term_id), int(r.instructor_id)): r for r in submissions.itertuples(index=False)}
availability = {(int(r.submission_id), int(r.slot_id)): r.kind for r in tables["availability_slots"].itertuples(index=False)}
allocations = tables["allocations"].merge(tables["time_slots"][["id", "weekday", "starts_at"]],
    left_on="start_slot_id", right_on="id", validate="many_to_one", suffixes=("", "_slot"))

def minutes(value):
    hh, mm = str(value).split(":")[:2]
    return int(hh) * 60 + int(mm)

def overlaps(start1, end1, start2, end2):
    return max(start1, start2) < min(end1, end2)


## Hard constraints

`check_allocation()` returns every reason in a stable order. The primary reason follows the CSV cases' intended priority. `fixture_group_ids` is a case-local override and never changes the shared source data. An unlisted availability slot is treated as unknown and blocked.

In [ ]:
PRIORITY = ["NO_WORKING_SLOT", "INVALID_DURATION", "AVAILABILITY_NOT_CONFIRMED",
            "STAFF_UNAVAILABLE", "ROOM_CONFLICT", "STAFF_CONFLICT", "GROUP_CONFLICT",
            "ROOM_TYPE_MISMATCH", "CAPACITY_SHORTAGE", "EQUIPMENT_SHORTAGE",
            "INELIGIBLE_INSTRUCTOR", "INVALID_REFERENCE"]

def check_allocation(term_id, section_id, requirement_id, instructor_id, room_id,
                     weekday, starts_at, ends_at, fixture_group_ids=None,
                     existing=allocations):
    term_id, section_id, requirement_id = map(int, (term_id, section_id, requirement_id))
    instructor_id, room_id, weekday = map(int, (instructor_id, room_id, weekday))
    reasons = set()
    if any(k not in d for k, d in [(section_id, sections), (requirement_id, requirements),
                                    (room_id, rooms), (instructor_id, accounts)]):
        return {"primary_result": "INVALID_REFERENCE", "reasons": ["INVALID_REFERENCE"], "slot_id": None}
    section, req, room = sections[section_id], requirements[requirement_id], rooms[room_id]
    if section["term_id"] != term_id or req["term_id"] != term_id or section["course_id"] != req["course_id"]:
        reasons.add("INVALID_REFERENCE")
    if (section_id, requirement_id, instructor_id) not in eligible:
        reasons.add("INELIGIBLE_INSTRUCTOR")
    start, end = minutes(starts_at), minutes(ends_at)
    matching = [(sid, row) for sid, row in slots.items()
                if int(row["term_id"]) == term_id and int(row["weekday"]) == weekday
                and minutes(row["starts_at"]) == start]
    slot_id = matching[0][0] if matching else None
    if not matching or start < 9 * 60:
        reasons.add("NO_WORKING_SLOT")
    if end - start != int(req["duration_minutes"]) or (matching and end != minutes(matching[0][1]["ends_at"])):
        reasons.add("INVALID_DURATION")
    submission = submission_by_staff.get((term_id, instructor_id))
    if submission is None or submission.state != "CONFIRMED":
        reasons.add("AVAILABILITY_NOT_CONFIRMED")
    elif slot_id is not None and availability.get((int(submission.id), slot_id)) not in ("AVAILABLE", "PREFERRED"):
        reasons.add("STAFF_UNAVAILABLE")
    if room["kind"] != req["required_room_kind"] or not bool(room["active"]):
        reasons.add("ROOM_TYPE_MISMATCH")
    if students_by_section.get(section_id, 0) > int(room["capacity"]):
        reasons.add("CAPACITY_SHORTAGE")
    for (req_id, equipment_id), quantity in required.items():
        if req_id == requirement_id and available.get((room_id, equipment_id), 0) < quantity:
            reasons.add("EQUIPMENT_SHORTAGE")
    candidate_groups = set(fixture_group_ids) if fixture_group_ids is not None else groups_by_section.get(section_id, set())
    for old in existing.itertuples(index=False):
        if int(old.term_id) != term_id or int(old.weekday) != weekday or int(old.section_id) == section_id:
            continue
        if not overlaps(start, end, minutes(old.starts_at), minutes(old.ends_at)):
            continue
        if int(old.room_id) == room_id: reasons.add("ROOM_CONFLICT")
        if int(old.instructor_id) == instructor_id: reasons.add("STAFF_CONFLICT")
        if candidate_groups & groups_by_section.get(int(old.section_id), set()): reasons.add("GROUP_CONFLICT")
    ordered = [reason for reason in PRIORITY if reason in reasons]
    return {"primary_result": ordered[0] if ordered else "FEASIBLE", "reasons": ordered,
            "slot_id": slot_id}


## Work through the supplied cases

The CSV's `student_count`, `room_capacity`, and equipment quantities are explanatory snapshots. The checker reads those facts from the related CSV tables. The group collision fixture temporarily shares a group with section 1.

In [ ]:
results = []
for c in cases.itertuples(index=False):
    fixture = None
    if c.case_id == "GROUP_COLLISION":
        fixture = (groups_by_section[int(c.section_id)] - {max(groups_by_section[int(c.section_id)])}) | {min(groups_by_section[1])}
    result = check_allocation(c.term_id, c.section_id, c.requirement_id, c.instructor_id,
                              c.room_id, c.weekday, c.starts_at, c.ends_at, fixture_group_ids=fixture)
    results.append({"case_id": c.case_id, "expected": c.expected_primary_result,
                    "actual": result["primary_result"], "all_reasons": result["reasons"],
                    "pass": c.expected_primary_result == result["primary_result"]})
case_results = pd.DataFrame(results)
display(case_results)
assert case_results["pass"].all(), case_results.loc[~case_results["pass"]]


## Generate feasible candidates and rank alternatives

The demo generates candidate combinations for one section and one weekly session. It excludes hard conflicts and prefers a `PREFERRED` availability slot, then an earlier time. For a full timetable solver, add weekly session instances, teacher workload, room closure/holiday tables, and an optimizer that chooses a compatible set *jointly*. This starter does not claim to solve the global optimization problem.

In [ ]:
def rank_candidates(section_id, requirement_id, limit=10):
    sec = sections[int(section_id)]
    term_id = int(sec["term_id"])
    staff = sorted({i for s, r, i in eligible if s == int(section_id) and r == int(requirement_id)})
    rows = []
    for instructor_id in staff:
        submission = submission_by_staff.get((term_id, instructor_id))
        for slot_id, slot in slots.items():
            if int(slot["term_id"]) != term_id: continue
            for room_id in rooms:
                result = check_allocation(term_id, section_id, requirement_id, instructor_id,
                    room_id, slot["weekday"], slot["starts_at"], slot["ends_at"])
                if result["primary_result"] != "FEASIBLE": continue
                kind = availability.get((int(submission.id), slot_id), "UNKNOWN") if submission else "UNKNOWN"
                rows.append({"section_id": section_id, "requirement_id": requirement_id,
                             "instructor_id": instructor_id, "room_id": room_id,
                             "slot_id": slot_id, "weekday": slot["weekday"],
                             "starts_at": slot["starts_at"], "availability": kind,
                             "preference_rank": 0 if kind == "PREFERRED" else 1})
    if not rows: return pd.DataFrame(columns=["section_id", "requirement_id", "instructor_id", "room_id", "slot_id", "weekday", "starts_at", "availability", "preference_rank"])
    return pd.DataFrame(rows).sort_values(["preference_rank", "weekday", "starts_at", "room_id"]).head(limit).reset_index(drop=True)

candidate_view = rank_candidates(section_id=2, requirement_id=1)
candidate_view


## Menna's next steps

1. Keep `check_allocation` as a testable hard-constraint baseline and add any approved rules from the project brief.
2. Model each required weekly session as a separate decision variable, then choose room, time and eligible instructor together. Re-check conflicts against both existing and newly selected allocations.
3. Add soft scores such as preferred availability and room utilization. Track every score's definition; hard conflicts must remain disqualifying.
4. Add term holidays and room closures when those CSVs are available. Use the published schedule version as an immutable input to change requests.
5. Feed reason codes and ranked options to Osama's API and Omar's Scheduler UI. Run the 11 fixture cases after each change.